# Intent assignment (EN): TF-IDF + ML models vs. LLM
Baseline: **TF-IDF + LinearSVC**. Compared with TF-IDF + LR / XGBoost / LightGBM / RandomForest and an LLM (labels only). Evaluated overall, per intent and per data slice (clean vs. typo-noisy, utterance length, Bitext style tags). Vietnamese is out of scope for now.

In [ ]:
import os, sys, subprocess
from pathlib import Path
# Needs only experiments/eval_intent_en.py + llm_classifier.py. Find them in the repo, else clone it (Kaggle).
NAME = 'services/intent_service/src/pipeline/experiments'
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / NAME).exists()), None)
if root is None:
    root = Path('/kaggle/working/repo') if Path('/kaggle').exists() else Path('repo')
    if not root.exists():
        subprocess.run(['git', 'clone', '-b', 'hung', 'https://github.com/DunnTapCodee/E-commerce-Support-Copilot.git', str(root)], check=True)
    # No clone available / files not pushed yet? Upload the two .py files next to this notebook instead.
if Path('/kaggle').exists() and (root / '.git').exists():  # Kaggle copy is disposable: always sync to the latest pushed code
    subprocess.run(['git', '-C', str(root), 'fetch', 'origin', 'hung'], check=True)
    subprocess.run(['git', '-C', str(root), 'reset', '--hard', 'FETCH_HEAD'], check=True)
EXP = root / NAME if (root / NAME).exists() else Path.cwd()
sys.path.insert(0, str(EXP)); os.chdir(root if (root / NAME).exists() else Path.cwd())
# %pip install -q xgboost lightgbm scikit-learn pandas pyarrow datasets httpx matplotlib
import pandas as pd, numpy as np, matplotlib.pyplot as plt
for m in ['eval_intent_en', 'llm_classifier']: sys.modules.pop(m, None)  # drop stale copies from earlier runs
import eval_intent_en as E
print('repo commit:', subprocess.run(['git', '-C', str(root), 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

## 1. Data
Uses `data/processed/module1/*_en.parquet` when built, otherwise raw Bitext from HuggingFace split by text group (no leakage). The test set is doubled with a typo-noisy copy.

In [ ]:
train, val, tests = E.load_en_data()
labels = sorted(train.intent.unique())
print(len(train), len(val), len(tests), len(labels), 'intents')
tests.dtype.value_counts()

## 2. ML models (run any cell on its own)
Each cell trains one model and stores its predictions in `preds`.

In [ ]:
preds, times = {}, {}
def run_model(key):
    m = E.build_models([key])[key]; m.fit(train, val)
    preds[m.name] = m.predict(tests.text); times[m.name] = m.train_time_s
    s, _ = E.score(tests.intent, preds[m.name], labels)
    print(m.name, f'train {m.train_time_s:.1f}s', {k: round(v, 4) for k, v in s.items()})

In [ ]:
run_model('linearsvc')   # baseline

In [ ]:
run_model('lr')

In [ ]:
run_model('sgd')

In [ ]:
run_model('ridge')

In [ ]:
run_model('pa')

In [ ]:
run_model('cnb')

In [ ]:
run_model('knn')

In [ ]:
run_model('extratrees')

In [ ]:
run_model('mlp')

In [ ]:
run_model('xgboost')

In [ ]:
run_model('lightgbm')

In [ ]:
run_model('rf')

### Tune hyper-parameters (validation set only)
Example: sweep LinearSVC `C`, then edit the model's default in `build_models` with the best value.

In [ ]:
E.tune('linearsvc', {'C': [.1, .25, .5, 1, 2]}, train, val)

In [ ]:
E.tune('lr', {'C': [1, 5, 10, 20]}, train, val)

## 3. LLM intent assignment
Set `LLM_API_KEY` (and optionally `LLM_MODEL`); host defaults to `https://api.xah.io/v1`. The LLM is run on a stratified sample (cached on disk) to limit cost; ML models are re-scored on the same sample for a fair comparison.

In [ ]:
# Key is never stored in the notebook: use Kaggle Secrets (name LLM_API_KEY), a local .env, or env vars.
try:
    from kaggle_secrets import UserSecretsClient
    for name in ['LLM_API_KEY', 'GEMINI_API_KEY']:
        try:
            os.environ.setdefault('LLM_API_KEY', UserSecretsClient().get_secret(name)); break
        except Exception as e:
            print(f'Kaggle secret {name!r} not available: {e}')
except ImportError:
    pass
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError:
    pass
os.environ.setdefault('LLM_API_BASE', 'https://generativelanguage.googleapis.com/v1beta/openai')  # or https://api.xah.io/v1
os.environ.setdefault('LLM_MODEL', 'gemini-3.8-flash')
from llm_classifier import LLMIntentClassifier
N_LLM = 200   # None = whole test set
sub = E.stratified_sample(tests, N_LLM)
assert os.getenv('LLM_API_KEY'), "No key: Kaggle > Add-ons > Secrets > add LLM_API_KEY and tick Attached for this notebook; or set os.environ['LLM_API_KEY']."
llm = LLMIntentClassifier(labels)
llm_pred = llm.predict(sub.text)
print(llm.model, 'invalid outputs:', np.mean(np.array(llm_pred) == 'invalid_llm_output'))

## 4. Comparison (all models on the same sample)

In [ ]:
comp = {k: pd.Series(v, index=tests.index).loc[sub.index].values for k, v in preds.items()}
comp[f'llm_{llm.model}'] = llm_pred
overall, per_intent, slices = E.evaluate(comp, sub, labels)
overall['train_time_s'] = overall.model.map(times)
overall

### Which data type is each model good at?

In [ ]:
slices.pivot_table(index=['slice','value'], columns='model', values='accuracy').round(3)

### Which intents? (F1 per intent, sorted by support; worst intents of the best ML model first)

In [ ]:
best_ml = overall[~overall.model.str.startswith('llm')].model.iloc[0]
table = per_intent.sort_values(best_ml)
table.round(2).style.background_gradient(cmap='RdYlGn', subset=[c for c in table.columns if c != 'support'], vmin=0, vmax=1)

In [ ]:
gap = (per_intent[f'llm_{llm.model}'] - per_intent[best_ml]).sort_values()
print('Intents where LLM is much worse than', best_ml); print(gap.head(8).round(2))
print('Intents where LLM is much better'); print(gap.tail(8).round(2))

In [ ]:
ax = overall.set_index('model')[['accuracy','macro_f1']].plot.bar(figsize=(9,4), ylim=(0,1)); ax.set_title('Overall'); plt.tight_layout(); plt.show()

In [ ]:
out = E.OUT; out.mkdir(parents=True, exist_ok=True)
overall.to_csv(out/'overall.csv', index=False); per_intent.to_csv(out/'per_intent_f1.csv'); slices.to_csv(out/'slices.csv', index=False)